# IsoRank and affinity-threshold sensitivity analysis

This notebook reproduces the modular argument-graph pipeline from `1.7_graph_construction_node_attribute_modular.ipynb` and evaluates two one-factor-at-a-time sweeps:

1. vary `ISORANK_ALPHA` while holding `MIN_SIMILARITY = 0.55`;
2. vary `MIN_SIMILARITY` while holding `ISORANK_ALPHA = 0.05`.

For every setting, the notebook reports sparse-affinity diagnostics and spectral-partition metrics. Embeddings and document graphs are computed once and reused.

In [1]:
from __future__ import annotations

import logging
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from IPython.display import display
from plotly.subplots import make_subplots

search_roots = (Path.cwd(), *Path.cwd().parents)
PROJECT_ROOT = next(
    root for root in search_roots
    if (root / "src" / "legal_data_doc_parts").is_dir()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from legal_data_doc_parts.argument_graph import (
    ArgumentGraphBuilder,
    GlobalSimilarityMatrixBuilder,
    LocalSpectralPartitioner,
    PartitionConfig,
    RelationAwareIsoRank,
    TransformerMeanPoolEmbedder,
    embed_node_attributes,
    load_and_validate_documents,
    locate_project_resource,
    mutual_knn_graph,
    validate_global_similarity,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("argument-graph-parameter-sensitivity")

RANDOM_SEED = 7
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_BATCH_SIZE = 64
ISORANK_TOL = 1e-5
ISORANK_MAX_ITER = 500
K_NEIGHBORS = 12

BASELINE_ISORANK_ALPHA = 0.05
BASELINE_MIN_SIMILARITY = 0.55
ISORANK_ALPHA_VALUES = (0.00, 0.05, 0.07, 0.10, 0.15, 0.2, 0.25, 0.40, 0.5)
MIN_SIMILARITY_VALUES = (0.55, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95, 0.98)

PARTITION_CONFIG = PartitionConfig(
    restart=0.15,
    pagerank_tolerance=1e-10,
    pagerank_max_iterations=500,
    star_radius=2,
    max_star_nodes=128,
    maximum_conductance=0.80,
    one_node_per_graph=True,
)

logger.info("PyTorch device: %s", DEVICE)

2026-11-23 06:07:34,563 | INFO | PyTorch device: cuda


In [2]:
paragraph_dataset_mini = [
    "116062204", "116075685", "116080900", "116084352", "116084886",
    "116099338", "116102314", "116105977", "116106432", "116109598",
    "116110555", "116116081", "116118921", "116120654", "116121714",
    "116125191", "116126076", "116127831", "116141779", "116142769",
    "116149190", "116150283", "116167789", "116168634", "116170057",
    "116170822", "116192026", "116194637", "116196887", "116197578",
    "116199866", "116204676", "116213690", "116214232", "116220471",
    "116221948", "116226431", "116229939", "116232414", "116233794",
    "116245112", "116246991", "116248468", "116252607", "116253514",
    "116256069", "116257159", "116259036", "116260246", "116261015",
    "116261263", "116270112", "116277632", "116278949", "116283678",
    "116285648", "116289180", "116308208", "116310154", "116320626",
    "116321603", "116336690", "116339406", "116344031", "116350541",
    "116350825", "116358099", "116359089", "116376162", "116377931",
    "116385451", "116398308", "116410200", "116410462", "116414218",
    "116432495", "116434259", "116442566", "116462199", "116588653",
    "116681882", "116764649", "116780853", "116790074", "116794466",
    "116972204", "117069850", "117125134", "117130242", "117230819",
    "117251852", "117420480", "117440182", "117456818", "117460145",
    "117464049", "117489745", "117516936", "117556435",
]

assert paragraph_dataset_mini
assert len(paragraph_dataset_mini) == len(set(paragraph_dataset_mini))
logger.info("Configured %d documents", len(paragraph_dataset_mini))

2026-11-23 06:07:34,674 | INFO | Configured 99 documents


## 1. Reading and validation

The data-loading path, validation rules, and document order match the modular source notebook.

In [3]:
DATA_DIR = locate_project_resource(Path("document_data_parsing"))
ENCODER_DIR = locate_project_resource(
    Path("bge_model/bge-small-uk-law-v1_1/final/encoder")
)
TOKENIZER_DIR = locate_project_resource(
    Path("bge_model/bge-small-uk-law-v1_1/final/tokenizer")
)

dataset = load_and_validate_documents(paragraph_dataset_mini, DATA_DIR)
doc_args = dataset.documents
arguments_df = dataset.arguments
relations_df = dataset.relations

dataset_summary = pd.Series(
    {
        "documents": len(doc_args),
        "arguments": len(arguments_df),
        "relations": len(relations_df),
        "premises": int((arguments_df["argument_type"] == "PREMISE").sum()),
        "claims": int((arguments_df["argument_type"] == "CLAIM").sum()),
        "duplicate_source_target_pairs": int(
            relations_df.duplicated(
                ["document_id", "source", "target"]
            ).sum()
        ),
    },
    name="count",
)
display(dataset_summary.to_frame())

,count
documents,99
arguments,3941
relations,2928
premises,2710
claims,1231
duplicate_source_target_pairs,15


## 2. Shared embeddings and document graphs

These expensive stages run once. Both parameter sweeps reuse the same node embeddings, relation channels, graph ordering, and partition configuration.

In [4]:
embedder = TransformerMeanPoolEmbedder.from_pretrained(
    ENCODER_DIR,
    TOKENIZER_DIR,
    device=DEVICE,
    batch_size=EMBEDDING_BATCH_SIZE,
)
embedded_nodes = embed_node_attributes(
    arguments_df,
    embedder,
    device=DEVICE,
)
text_embeddings = embedded_nodes.text_embeddings
section_embeddings = embedded_nodes.section_embeddings
section_present = embedded_nodes.section_present

logger.info(
    "Encoded %d arguments into %d dimensions",
    text_embeddings.shape[0],
    text_embeddings.shape[1],
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

2026-11-23 06:07:46,068 | INFO | Encoded 3941 arguments into 384 dimensions


In [5]:
graph_collection = ArgumentGraphBuilder(device=DEVICE).build(
    arguments_df,
    relations_df,
    embedded_nodes,
    paragraph_dataset_mini,
)
case_data = graph_collection.cases
cases = graph_collection.graphs
graph_sizes = graph_collection.graph_sizes
offsets = graph_collection.offsets
channel_names = graph_collection.channel_names
node_graph_index = arguments_df["graph_index"].to_numpy(
    dtype=np.int64,
    copy=True,
)

graph_summary = pd.DataFrame(
    {
        "document_id": paragraph_dataset_mini,
        "nodes": graph_sizes,
        "relations": [graph.number_of_edges() for graph in cases],
        "isolated_nodes": [
            sum(graph.degree(node) == 0 for node in graph.nodes)
            for graph in cases
        ],
    }
)
display(graph_summary.describe().round(2))

,nodes,relations,isolated_nodes
count,99.00,99.00,99.00
mean,39.81,29.58,10.12
std,21.55,13.55,11.70
min,1.00,0.00,0.00
25%,28.00,22.00,3.00
50%,37.00,30.00,7.00
75%,47.50,37.00,11.50
max,132.00,66.00,67.00


## 3. Spectral-partition evaluation

Each run constructs the mutual-kNN affinity graph, applies isolate repair, runs the same local spectral partitioner, validates complete disjoint coverage, and records graph, cluster, conductance, PageRank, and runtime metrics.

In [6]:
PARTITION_METRIC_COLUMNS = [
    "affinity_edges",
    "mean_affinity_degree",
    "isolates_before_repair",
    "fallback_edges",
    "isolates_after_repair",
    "clusters",
    "multi_node_clusters",
    "singletons",
    "largest_cluster",
    "mean_cluster_size",
    "median_cluster_size",
    "mean_conductance",
    "median_conductance",
    "accepted_sweeps",
    "mean_candidate_nodes",
    "mean_candidate_graphs",
    "mean_pagerank_iterations",
    "affinity_seconds",
    "partition_seconds",
]


def _assert_complete_partition(clusters, node_count):
    assigned_nodes = np.concatenate(
        [
            np.asarray(cluster["nodes"], dtype=np.int64)
            for cluster in clusters
        ]
    )
    if assigned_nodes.size != node_count:
        raise AssertionError(
            f"Expected {node_count} memberships, got {assigned_nodes.size}"
        )
    if not np.array_equal(
        np.sort(assigned_nodes),
        np.arange(node_count, dtype=np.int64),
    ):
        raise AssertionError(
            "Partition must cover every node exactly once"
        )


def collect_partition_metrics(similarity_matrix, minimum_similarity):
    affinity_started = time.perf_counter()
    affinity_result = mutual_knn_graph(
        similarity_matrix,
        neighbors=K_NEIGHBORS,
        minimum_similarity=minimum_similarity,
    )
    affinity_seconds = time.perf_counter() - affinity_started
    affinity_graph = affinity_result.adjacency

    weighted_degree = np.asarray(
        affinity_graph.sum(axis=1)
    ).ravel()
    isolated_after_repair = int(np.count_nonzero(weighted_degree == 0))
    edge_rows, edge_columns = affinity_graph.nonzero()
    same_graph_edges = int(
        np.count_nonzero(
            node_graph_index[edge_rows]
            == node_graph_index[edge_columns]
        )
    )

    if affinity_graph.nnz == 0:
        raise AssertionError("Affinity graph has no edges")
    if (affinity_graph - affinity_graph.T).nnz:
        raise AssertionError("Affinity graph is not symmetric")
    if same_graph_edges:
        raise AssertionError(
            f"Affinity graph contains {same_graph_edges} within-graph edges"
        )
    if isolated_after_repair:
        logger.warning(
            "Affinity graph retains %d isolates after repair",
            isolated_after_repair,
        )

    partition_started = time.perf_counter()
    partitioner = LocalSpectralPartitioner(
        node_graph_index,
        config=PARTITION_CONFIG,
    )
    clusters, diagnostics = partitioner.partition(
        affinity_graph,
        similarity_matrix,
    )
    partition_seconds = time.perf_counter() - partition_started
    _assert_complete_partition(clusters, similarity_matrix.shape[0])

    cluster_sizes = diagnostics["size"].to_numpy(dtype=np.int64)
    conductance = diagnostics["conductance"].replace(
        [np.inf, -np.inf],
        np.nan,
    )
    metrics = {
        "minimum_similarity": float(minimum_similarity),
        "partitioned_nodes": int(cluster_sizes.sum()),
        "affinity_edges": int(affinity_graph.nnz // 2),
        "mean_affinity_degree": float(
            affinity_graph.nnz / affinity_graph.shape[0]
        ),
        "isolates_before_repair": int(
            affinity_result.isolated_before_repair
        ),
        "fallback_edges": int(affinity_result.fallback_edges),
        "isolates_after_repair": isolated_after_repair,
        "clusters": int(len(clusters)),
        "multi_node_clusters": int(np.count_nonzero(cluster_sizes > 1)),
        "singletons": int(np.count_nonzero(cluster_sizes == 1)),
        "largest_cluster": int(cluster_sizes.max()),
        "mean_cluster_size": float(cluster_sizes.mean()),
        "median_cluster_size": float(np.median(cluster_sizes)),
        "mean_conductance": float(conductance.mean()),
        "median_conductance": float(conductance.median()),
        "accepted_sweeps": int(
            diagnostics["accepted_sweep"].astype(bool).sum()
        ),
        "mean_candidate_nodes": float(
            diagnostics["candidate_nodes"].mean()
        ),
        "mean_candidate_graphs": float(
            diagnostics["candidate_graphs"].mean()
        ),
        "mean_pagerank_iterations": float(
            diagnostics["pagerank_iterations"].mean()
        ),
        "minimum_edge_weight": float(affinity_graph.data.min()),
        "maximum_edge_weight": float(affinity_graph.data.max()),
        "affinity_seconds": float(affinity_seconds),
        "partition_seconds": float(partition_seconds),
    }
    return metrics, clusters, diagnostics


def make_sensitivity_figure(results, parameter, title):
    parameter_label = {
        "isorank_alpha": "IsoRank alpha",
        "minimum_similarity": "Minimum similarity",
    }[parameter]
    colors = {
        "clusters": "#2563EB",
        "multi_node_clusters": "#16A34A",
        "singletons": "#DC2626",
        "largest_cluster": "#7C3AED",
        "mean_cluster_size": "#0891B2",
        "median_cluster_size": "#EA580C",
        "mean_conductance": "#0F766E",
        "median_conductance": "#BE123C",
        "affinity_edges": "#475569",
        "isolates_before_repair": "#D97706",
        "fallback_edges": "#9333EA",
    }
    figure = make_subplots(
        rows=2,
        cols=2,
        specs=[[{}, {}], [{}, {"secondary_y": True}]],
        subplot_titles=(
            "Cluster counts",
            "Cluster size",
            "Conductance",
            "Affinity sparsification",
        ),
    )

    for metric, name in [
        ("clusters", "All clusters"),
        ("multi_node_clusters", "Multi-node clusters"),
        ("singletons", "Singletons"),
    ]:
        figure.add_trace(
            go.Scatter(
                x=results[parameter],
                y=results[metric],
                mode="lines+markers",
                name=name,
                line={"color": colors[metric]},
            ),
            row=1,
            col=1,
        )

    for metric, name in [
        ("largest_cluster", "Largest cluster"),
        ("mean_cluster_size", "Mean cluster size"),
        ("median_cluster_size", "Median cluster size"),
    ]:
        figure.add_trace(
            go.Scatter(
                x=results[parameter],
                y=results[metric],
                mode="lines+markers",
                name=name,
                line={"color": colors[metric]},
            ),
            row=1,
            col=2,
        )

    for metric, name in [
        ("mean_conductance", "Mean conductance"),
        ("median_conductance", "Median conductance"),
    ]:
        figure.add_trace(
            go.Scatter(
                x=results[parameter],
                y=results[metric],
                mode="lines+markers",
                name=name,
                line={"color": colors[metric]},
            ),
            row=2,
            col=1,
        )

    figure.add_trace(
        go.Scatter(
            x=results[parameter],
            y=results["affinity_edges"],
            mode="lines+markers",
            name="Affinity edges",
            line={"color": colors["affinity_edges"]},
        ),
        row=2,
        col=2,
        secondary_y=False,
    )
    for metric, name in [
        ("isolates_before_repair", "Isolates before repair"),
        ("fallback_edges", "Fallback edges"),
    ]:
        figure.add_trace(
            go.Scatter(
                x=results[parameter],
                y=results[metric],
                mode="lines+markers",
                name=name,
                line={"color": colors[metric], "dash": "dash"},
            ),
            row=2,
            col=2,
            secondary_y=True,
        )

    for row in (1, 2):
        for column in (1, 2):
            figure.update_xaxes(
                title_text=parameter_label,
                row=row,
                col=column,
            )
    figure.update_yaxes(title_text="Count", row=1, col=1)
    figure.update_yaxes(title_text="Nodes", row=1, col=2)
    figure.update_yaxes(title_text="Conductance", row=2, col=1)
    figure.update_yaxes(
        title_text="Affinity edges",
        row=2,
        col=2,
        secondary_y=False,
    )
    figure.update_yaxes(
        title_text="Isolates / fallback edges",
        row=2,
        col=2,
        secondary_y=True,
    )
    figure.update_layout(
        title={"text": title, "x": 0.5, "xanchor": "center"},
        template="plotly_white",
        height=820,
        hovermode="x unified",
        legend={"orientation": "h", "y": -0.12},
        margin={"l": 60, "r": 60, "t": 90, "b": 130},
    )
    return figure

## 4. IsoRank alpha sweep

Only the relation-propagation weight changes. Each alpha value rebuilds the global similarity matrix `H`; affinity construction and partitioning use the baseline minimum similarity of 0.55.

In [7]:
alpha_rows = []
baseline_H = None
baseline_h_summary = None

for isorank_alpha in ISORANK_ALPHA_VALUES:
    logger.info(
        "Alpha sweep: alpha=%.2f, minimum_similarity=%.2f",
        isorank_alpha,
        BASELINE_MIN_SIMILARITY,
    )
    aligner = RelationAwareIsoRank(
        channel_names,
        alpha=isorank_alpha,
        tolerance=ISORANK_TOL,
        max_iterations=ISORANK_MAX_ITER,
    )
    alignment_started = time.perf_counter()
    H, alignment_diagnostics = GlobalSimilarityMatrixBuilder(
        aligner,
        device=DEVICE,
    ).build(case_data, offsets)
    alignment_seconds = time.perf_counter() - alignment_started
    h_summary = validate_global_similarity(
        H,
        alignment_diagnostics,
        offsets,
    )

    metrics, _, _ = collect_partition_metrics(
        H,
        BASELINE_MIN_SIMILARITY,
    )
    metrics.update(
        {
            "isorank_alpha": float(isorank_alpha),
            "alignment_seconds": float(alignment_seconds),
            "h_mean_positive": float(h_summary["mean_positive"]),
            "h_median_row_maximum": float(
                h_summary["median_row_maximum"]
            ),
            "max_alignment_iterations": int(
                h_summary["max_alignment_iterations"]
            ),
            "max_alignment_delta": float(h_summary["max_final_delta"]),
        }
    )
    alpha_rows.append(metrics)

    if np.isclose(isorank_alpha, BASELINE_ISORANK_ALPHA):
        baseline_H = H
        baseline_h_summary = h_summary.copy()

if baseline_H is None or baseline_h_summary is None:
    raise AssertionError("Baseline alpha was not evaluated")

alpha_results = (
    pd.DataFrame(alpha_rows)
    .sort_values("isorank_alpha")
    .reset_index(drop=True)
)
alpha_display_columns = [
    "isorank_alpha",
    "minimum_similarity",
    "h_mean_positive",
    "h_median_row_maximum",
    "max_alignment_iterations",
    "alignment_seconds",
    *PARTITION_METRIC_COLUMNS,
]
display(alpha_results[alpha_display_columns].round(4))
del H, alignment_diagnostics

2026-11-23 06:07:55,474 | INFO | Alpha sweep: alpha=0.00, minimum_similarity=0.55
2026-11-23 06:07:56,375 | INFO | Aligned 500/4851 graph pairs
2026-11-23 06:07:57,184 | INFO | Aligned 1000/4851 graph pairs
2026-11-23 06:07:58,018 | INFO | Aligned 1500/4851 graph pairs
2026-11-23 06:07:58,820 | INFO | Aligned 2000/4851 graph pairs
2026-11-23 06:07:59,635 | INFO | Aligned 2500/4851 graph pairs
2026-11-23 06:08:00,441 | INFO | Aligned 3000/4851 graph pairs
2026-11-23 06:08:01,244 | INFO | Aligned 3500/4851 graph pairs
2026-11-23 06:08:02,030 | INFO | Aligned 4000/4851 graph pairs
2026-11-23 06:08:02,824 | INFO | Aligned 4500/4851 graph pairs
2026-11-23 06:08:03,395 | INFO | Aligned 4851/4851 graph pairs
2026-11-23 06:08:10,841 | INFO | Alpha sweep: alpha=0.05, minimum_similarity=0.55
2026-11-23 06:08:12,435 | INFO | Aligned 500/4851 graph pairs
2026-11-23 06:08:14,017 | INFO | Aligned 1000/4851 graph pairs
2026-11-23 06:08:15,590 | INFO | Aligned 1500/4851 graph pairs
2026-11-23 06:08:17

,isorank_alpha,minimum_similarity,h_mean_positive,h_median_row_maximum,max_alignment_iterations,alignment_seconds,affinity_edges,mean_affinity_degree,isolates_before_repair,fallback_edges,...,mean_cluster_size,median_cluster_size,mean_conductance,median_conductance,accepted_sweeps,mean_candidate_nodes,mean_candidate_graphs,mean_pagerank_iterations,affinity_seconds,partition_seconds
0,0.00,0.55,0.7488,0.9384,1,8.0049,10696,5.4281,426,426,...,3.3314,1.0,0.7470,1.0,541,5.2992,3.8005,21.0964,0.1224,6.1822
1,0.05,0.55,0.7355,0.9266,4,15.2583,10701,5.4306,361,361,...,3.5156,1.0,0.7249,1.0,530,5.6289,4.0839,22.0384,0.0837,6.0755
2,0.07,0.55,0.7300,0.9237,4,15.1998,10786,5.4737,350,350,...,3.5345,1.0,0.7263,1.0,524,5.6251,4.0816,21.6251,0.0764,6.0457
3,0.10,0.55,0.7215,0.9196,5,17.4487,10753,5.4570,360,360,...,3.5697,1.0,0.7238,1.0,528,5.7301,4.1313,22.2591,0.0820,6.0151
4,0.15,0.55,0.7068,0.9137,6,19.7759,10517,5.3372,407,407,...,3.6189,1.0,0.7103,1.0,532,5.7805,4.1414,22.4141,0.0801,5.9793
5,0.20,0.55,0.6914,0.9076,7,21.5175,10193,5.1728,484,484,...,3.4969,1.0,0.7137,1.0,537,5.5448,3.9805,20.8509,0.0855,5.8854
6,0.25,0.55,0.6752,0.9003,8,23.7169,9827,4.9871,573,573,...,3.4299,1.0,0.7139,1.0,544,5.4656,3.9191,20.6101,0.0897,7.0225
7,0.40,0.55,0.6211,0.8766,12,33.0307,8481,4.3040,1026,1023,...,3.3118,1.0,0.7053,1.0,565,5.5958,3.8756,19.0538,0.1261,6.5092
8,0.50,0.55,0.5793,0.8580,16,40.0770,7770,3.9432,1499,1492,...,3.1808,1.0,0.7257,1.0,524,5.9685,3.8959,18.0048,0.1396,6.2334


In [8]:
alpha_sensitivity_figure = make_sensitivity_figure(
    alpha_results,
    "isorank_alpha",
    (
        "Spectral partition sensitivity to IsoRank alpha "
        f"(minimum similarity = {BASELINE_MIN_SIMILARITY:.2f})"
    ),
)
alpha_sensitivity_figure.show()

## 5. Minimum-similarity sweep

The baseline matrix `H` from `ISORANK_ALPHA = 0.05` is reused. Only the threshold passed to mutual-kNN affinity construction changes, so this sweep isolates sparsification and partitioning effects.

In [9]:
similarity_rows = []

for minimum_similarity in MIN_SIMILARITY_VALUES:
    logger.info(
        "Similarity sweep: alpha=%.2f, minimum_similarity=%.2f",
        BASELINE_ISORANK_ALPHA,
        minimum_similarity,
    )
    metrics, _, _ = collect_partition_metrics(
        baseline_H,
        minimum_similarity,
    )
    metrics.update(
        {
            "isorank_alpha": BASELINE_ISORANK_ALPHA,
            "h_mean_positive": float(
                baseline_h_summary["mean_positive"]
            ),
            "h_median_row_maximum": float(
                baseline_h_summary["median_row_maximum"]
            ),
        }
    )
    similarity_rows.append(metrics)

similarity_results = (
    pd.DataFrame(similarity_rows)
    .sort_values("minimum_similarity")
    .reset_index(drop=True)
)
similarity_display_columns = [
    "minimum_similarity",
    "isorank_alpha",
    "h_mean_positive",
    "h_median_row_maximum",
    *PARTITION_METRIC_COLUMNS,
]
display(similarity_results[similarity_display_columns].round(4))

baseline_alpha_row = alpha_results.loc[
    np.isclose(
        alpha_results["isorank_alpha"],
        BASELINE_ISORANK_ALPHA,
    )
].iloc[0]
baseline_similarity_row = similarity_results.loc[
    np.isclose(
        similarity_results["minimum_similarity"],
        BASELINE_MIN_SIMILARITY,
    )
].iloc[0]
baseline_check_metrics = [
    "affinity_edges",
    "clusters",
    "multi_node_clusters",
    "singletons",
    "largest_cluster",
    "mean_conductance",
    "median_conductance",
]
np.testing.assert_allclose(
    baseline_alpha_row[baseline_check_metrics].to_numpy(dtype=float),
    baseline_similarity_row[baseline_check_metrics].to_numpy(dtype=float),
    rtol=0.0,
    atol=1e-12,
)
print("Baseline cross-check passed for both independent sweeps")

2026-11-23 06:12:24,601 | INFO | Similarity sweep: alpha=0.05, minimum_similarity=0.55
2026-11-23 06:12:31,346 | INFO | Similarity sweep: alpha=0.05, minimum_similarity=0.65
2026-11-23 06:12:31,434 | WARNING | Affinity graph retains 1 isolates after repair
2026-11-23 06:12:37,585 | INFO | Similarity sweep: alpha=0.05, minimum_similarity=0.70
2026-11-23 06:12:37,677 | WARNING | Affinity graph retains 1 isolates after repair
2026-11-23 06:12:43,801 | INFO | Similarity sweep: alpha=0.05, minimum_similarity=0.75
2026-11-23 06:12:43,900 | WARNING | Affinity graph retains 3 isolates after repair
2026-11-23 06:12:50,039 | INFO | Similarity sweep: alpha=0.05, minimum_similarity=0.80
2026-11-23 06:12:50,129 | WARNING | Affinity graph retains 48 isolates after repair
2026-11-23 06:12:56,339 | INFO | Similarity sweep: alpha=0.05, minimum_similarity=0.85
2026-11-23 06:12:56,427 | WARNING | Affinity graph retains 252 isolates after repair
2026-11-23 06:13:02,697 | INFO | Similarity sweep: alpha=0.0

,minimum_similarity,isorank_alpha,h_mean_positive,h_median_row_maximum,affinity_edges,mean_affinity_degree,isolates_before_repair,fallback_edges,isolates_after_repair,clusters,...,mean_cluster_size,median_cluster_size,mean_conductance,median_conductance,accepted_sweeps,mean_candidate_nodes,mean_candidate_graphs,mean_pagerank_iterations,affinity_seconds,partition_seconds
0,0.55,0.05,0.7355,0.9266,10701,5.4306,361,361,0,1121,...,3.5156,1.0,0.7249,1.0000,530,5.6289,4.0839,22.0384,0.0988,6.6359
1,0.65,0.05,0.7355,0.9266,10700,5.4301,361,360,1,1121,...,3.5156,1.0,0.7251,1.0000,529,5.6262,4.0812,22.0393,0.0826,6.1438
2,0.70,0.05,0.7355,0.9266,10700,5.4301,361,360,1,1121,...,3.5156,1.0,0.7251,1.0000,529,5.6262,4.0812,22.0393,0.0865,6.1179
3,0.75,0.05,0.7355,0.9266,10699,5.4296,363,360,3,1122,...,3.5125,1.0,0.7257,1.0000,528,5.6230,4.0784,22.0258,0.0947,6.1314
4,0.80,0.05,0.7355,0.9266,10636,5.3976,385,337,48,1159,...,3.4003,1.0,0.7278,1.0000,521,5.4202,3.9370,20.9638,0.0857,6.2032
5,0.85,0.05,0.7355,0.9266,10242,5.1977,516,264,252,1297,...,3.0386,1.0,0.7182,1.0000,500,4.7301,3.4726,17.5952,0.0834,6.2631
6,0.90,0.05,0.7355,0.9266,7815,3.9660,1228,138,1090,1892,...,2.0830,1.0,0.6746,1.0000,391,2.8436,2.2701,9.0217,0.1027,6.7605
7,0.95,0.05,0.7355,0.9266,1904,0.9663,2919,13,2906,3221,...,1.2235,1.0,0.3462,0.1105,234,1.2990,1.2428,2.4561,0.1471,8.1940
8,0.98,0.05,0.7355,0.9266,188,0.0954,3765,0,3765,3832,...,1.0284,1.0,0.1796,0.0000,57,1.0308,1.0287,1.1336,0.1794,8.9560


Baseline cross-check passed for both independent sweeps


In [10]:
minimum_similarity_figure = make_sensitivity_figure(
    similarity_results,
    "minimum_similarity",
    (
        "Spectral partition sensitivity to minimum similarity "
        f"(IsoRank alpha = {BASELINE_ISORANK_ALPHA:.2f})"
    ),
)
minimum_similarity_figure.show()

### Outputs

- `alpha_results`: one row per `ISORANK_ALPHA` value, with alignment, affinity, and spectral-partition metrics.
- `similarity_results`: one row per minimum-similarity threshold, using the baseline global similarity matrix.
- `alpha_sensitivity_figure` and `minimum_similarity_figure`: interactive Plotly comparisons of cluster counts, cluster sizes, conductance, and affinity sparsification.

The shared baseline row is evaluated in both sweeps and asserted to match.